# 🏭 Módulo 18 — Agentes con Microsoft Foundry

> **Objetivo:** Aprender a crear agentes conectados a Microsoft Foundry, tanto con inferencia directa (`FoundryChatClient`) como con agentes gestionados en el servicio (`FoundryAgent`).

📚 **Doc oficial:** [Microsoft Foundry Provider (Python)](https://learn.microsoft.com/en-us/agent-framework/agents/providers/microsoft-foundry?pivots=programming-language-python)

## ¿Qué es Microsoft Foundry?

Microsoft Foundry es una plataforma completamente gestionada para construir, desplegar y escalar agentes de IA.
Soporta múltiples modelos del catálogo de Foundry y proporciona herramientas integradas como
búsqueda web, intérprete de código, búsqueda de archivos, servidores MCP y más.

## Dos patrones para usar Foundry con Agent Framework

| Patrón | Clase | Cuándo usar |
|--------|-------|-------------|
| **Inferencia directa** | `Agent(client=FoundryChatClient(...))` | Tu app controla instrucciones, tools y sesiones |
| **Agente gestionado** | `FoundryAgent(...)` | El agente vive en Foundry (Prompt Agent o Hosted Agent) |

```
┌─────────────────────────────────────────────────────┐
│              Microsoft Foundry                       │
│  ┌──────────────┐    ┌──────────────────────────┐   │
│  │  Modelo       │    │  Agente Gestionado        │   │
│  │  (gpt-4o, etc)│    │  (Prompt / Hosted Agent)  │   │
│  └──────┬───────┘    └───────────┬──────────────┘   │
│         │                        │                   │
└─────────┼────────────────────────┼───────────────────┘
          │                        │
  FoundryChatClient          FoundryAgent
          │                        │
    Agent(client=...)        agent.run(...)
```

## Prerrequisitos

1. Un proyecto de Microsoft Foundry con un modelo desplegado (ej. `gpt-4o`)
2. **`az login` ejecutado** — Foundry requiere autenticación por token (API key NO es soportada por el SDK)
3. Variables de entorno configuradas (ver celda de setup)

## ⚙️ Setup

Instala las dependencias necesarias y configura las variables de entorno.

In [1]:
# Instalar dependencias de Foundry (ejecutar solo la primera vez)
# !pip install agent-framework-foundry azure-identity

In [2]:
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from helpers.config import (
    create_foundry_client,
    get_foundry_endpoint,
    get_foundry_model,
    get_foundry_agent_name,
)

print(f"📌 Endpoint: {get_foundry_endpoint()}")
print(f"📌 Modelo:   {get_foundry_model()}")
print(f"📌 Agente:   {get_foundry_agent_name()}")
print("\n✅ Config cargada desde .env vía helpers/config.py")

C:\Users\jreyesleger\AppData\Roaming\Python\Python312\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
C:\Users\jreyesleger\AppData\Roaming\Python\Python312\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


📌 Endpoint: https://se-openia-dev.services.ai.azure.com/api/projects/se-openia-dev-project
📌 Modelo:   gpt-4o
📌 Agente:   mi-agente-taller

✅ Config cargada desde .env vía helpers/config.py


## 1️⃣ Inferencia directa con `FoundryChatClient`

`FoundryChatClient` conecta a un modelo desplegado en un proyecto Foundry usando el endpoint de Responses.
Tu app controla las instrucciones, tools y el loop de conversación.

Este es el patrón más flexible — ideal cuando quieres control total sobre el comportamiento del agente.

In [3]:
from agent_framework import Agent

# create_foundry_client() usa AzureCliCredential automáticamente
foundry_client = create_foundry_client()

agent = Agent(
    client=foundry_client,
    name="AgenteFoundry",
    instructions="Eres un asistente experto en Azure. Responde siempre en español y de forma concisa.",
)

response = await agent.run("¿Qué es Microsoft Foundry y para qué sirve?")
print(f"🤖 [{agent.name}]: {response.text}")

🤖 [AgenteFoundry]: Microsoft Foundry es una iniciativa de Microsoft orientada a fomentar la innovación y el desarrollo ágil de soluciones tecnológicas. Funciona como un laboratorio de innovación donde equipos multidisciplinarios, a menudo compuestos por empleados, estudiantes y pasantes, trabajan en proyectos experimentales para crear prototipos, probar nuevas ideas y desarrollar aplicaciones o servicios.

El objetivo principal de Foundry es acelerar el desarrollo de productos, probar tecnologías emergentes y fomentar la creatividad en un entorno colaborativo. Es común que estos proyectos se enfoquen en áreas como inteligencia artificial, desarrollo de aplicaciones, integración con servicios de Azure y otros productos del ecosistema Microsoft.


### 1.1 — Con herramientas integradas de Foundry

`FoundryChatClient` expone métodos estáticos para cada herramienta hospedada en Foundry.
No necesitas una instancia para crearlas — son class methods.

In [4]:
from agent_framework.foundry import FoundryChatClient

# Ejemplo: Agente con búsqueda web y code interpreter
agent_con_tools = Agent(
    client=foundry_client,
    name="AgenteConTools",
    instructions=(
        "Eres un asistente de investigación. Usa la búsqueda web para encontrar información actualizada "
        "y el intérprete de código para hacer cálculos cuando sea necesario. Responde en español."
    ),
    tools=[
        FoundryChatClient.get_web_search_tool(),
        FoundryChatClient.get_code_interpreter_tool(),
    ],
)

response = await agent_con_tools.run("¿Cuántos días faltan para el 1 de enero de 2027?")
print(f"🤖 [{agent_con_tools.name}]: {response.text}")

Can't parse tool.
Can't parse tool.


🤖 [AgenteConTools]: Faltan 217 días para el 1 de enero de 2027. ⏳


### 1.2 — Con tools locales (funciones Python)

Puedes combinar herramientas hospedadas de Foundry con funciones Python locales.

In [5]:
from typing import Annotated
from agent_framework import tool


@tool
def obtener_clima(
    ciudad: Annotated[str, "Nombre de la ciudad"],
) -> str:
    """Obtiene el clima actual de una ciudad (simulado)."""
    climas = {
        "madrid": "☀️ 28°C, soleado",
        "cdmx": "🌧️ 18°C, lluvia ligera",
        "bogota": "⛅ 15°C, parcialmente nublado",
    }
    return climas.get(ciudad.lower(), f"No tengo datos del clima para {ciudad}")


agent_mixto = Agent(
    client=foundry_client,
    name="AgenteMixto",
    instructions="Eres un asistente que ayuda con información del clima. Responde en español.",
    tools=[obtener_clima],
)

response = await agent_mixto.run("¿Cómo está el clima en Madrid?")
print(f"🤖 [{agent_mixto.name}]: {response.text}")

🤖 [AgenteMixto]: Actualmente en Madrid hace 28°C y está soleado. ☀️


## 2️⃣ Agente gestionado con `FoundryAgent`

`FoundryAgent` conecta a un agente que **ya existe en Foundry** (creado en el portal o vía API).
Las instrucciones y herramientas viven en Foundry, no en tu código Python.

### Crear el agente en Foundry primero

Antes de ejecutar esta celda, crea un agente en el [portal de Foundry](https://ai.azure.com):

1. Ve a tu proyecto → **Agentes** → **Crear agente**
2. Dale nombre (ej. `mi-agente-taller`), instrucciones y modelo
3. Opcionalmente agrega herramientas (búsqueda web, code interpreter, etc.)
4. **Publica** el agente para obtener un nombre y versión

> 💡 También puedes crear el agente por código con `azure-ai-projects` SDK (ver celda siguiente).

In [6]:
from azure.ai.projects.aio import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()

project = AIProjectClient(
    endpoint=get_foundry_endpoint(),
    credential=credential,
)

agent_def = await project.agents.create_version(
    agent_name=get_foundry_agent_name(),
    definition=PromptAgentDefinition(
        model=get_foundry_model(),
        instructions=(
            "Eres un asistente especializado en Azure y Microsoft Foundry. "
            "Responde siempre en español de forma clara y concisa."
        ),
    ),
)
print(f"✅ Agente creado: {agent_def.name} v{agent_def.version}")

✅ Agente creado: mi-agente-taller v1


### 2.1 — Conectar y usar el agente gestionado

In [7]:
from agent_framework.foundry import FoundryAgent

foundry_agent = FoundryAgent(
    project_endpoint=get_foundry_endpoint(),
    agent_name=get_foundry_agent_name(),
    agent_version=str(agent_def.version),
    credential=credential,
)

print(f"✅ Conectado al agente '{foundry_agent.name}' en Foundry")

response = await foundry_agent.run("¿Qué servicios de IA ofrece Microsoft Foundry?")
print(f"\n🤖 [{foundry_agent.name}]: {response.text}")

✅ Conectado al agente 'mi-agente-taller' en Foundry

🤖 [mi-agente-taller]: Microsoft Foundry, aunque no es directamente conocida como un proveedor de servicios de IA en el mismo sentido que Azure, es un laboratorio de innovación y desarrollo que se enfoca en crear soluciones tecnológicas novedosas utilizando herramientas de Microsoft, incluidas capacidades de inteligencia artificial. Foundry trabaja en colaboración con equipos internos y socios para prototipar, experimentar y construir proyectos de software avanzados.

En términos generales, si bien Foundry no ofrece servicios de IA comercializados como lo hace Azure, sus desarrollos aprovechan las herramientas y servicios de IA dentro del ecosistema Microsoft, entre ellas:

1. **Azure Cognitive Services**: APIs para visión, lenguaje, habla y toma de decisiones automatizadas que pueden ser integradas en proyectos.
2. **Azure Machine Learning**: Plataforma para crear y desplegar modelos de aprendizaje automático personalizados.
3. **Bot

### 2.2 — Conversación multi-turno con sesiones

`FoundryAgent` mantiene el historial automáticamente usando sesiones del servicio.
El agente recuerda lo que se dijo en turnos anteriores.

In [8]:
from agent_framework import AgentSession

# Crear una sesión para mantener el contexto entre turnos
session = AgentSession()

preguntas = [
    "¿Qué es un Prompt Agent en Foundry?",
    "¿Y cuál es la diferencia con un Hosted Agent?",
    "Dame un ejemplo práctico de cuándo usar cada uno.",
]

for pregunta in preguntas:
    print(f"👤 Usuario: {pregunta}")
    response = await foundry_agent.run(pregunta, session=session)
    # Truncar para legibilidad
    texto = response.text[:200] + "..." if len(response.text) > 200 else response.text
    print(f"🤖 [{foundry_agent.name}]: {texto}\n")

print("✅ Conversación multi-turno completada.")

👤 Usuario: ¿Qué es un Prompt Agent en Foundry?
🤖 [mi-agente-taller]: En **Palantir Foundry**, un **Prompt Agent** es un componente diseñado para interactuar con los usuarios a través de lenguaje natural, facilitando la colaboración entre humanos y sistemas basados en i...

👤 Usuario: ¿Y cuál es la diferencia con un Hosted Agent?
🤖 [mi-agente-taller]: En **Palantir Foundry**, tanto los **Prompt Agents** como los **Hosted Agents** son componentes que permiten automatizar tareas o procesos, pero tienen diferencias clave en su naturaleza y propósito:
...

👤 Usuario: Dame un ejemplo práctico de cuándo usar cada uno.
🤖 [mi-agente-taller]: Claro, aquí tienes ejemplos prácticos de uso tanto para un **Prompt Agent** como para un **Hosted Agent** en **Palantir Foundry**:

---

### **Ejemplo de uso de un Prompt Agent**:  
**Escenario**: Un ...

✅ Conversación multi-turno completada.


## 3️⃣ Comparación lado a lado

Veamos la diferencia práctica entre los dos patrones con la misma pregunta.

In [9]:
pregunta = "Explica en una oración qué es un agente de IA."

# Patrón 1: Inferencia directa (instrucciones en código)
agente_local = Agent(
    client=foundry_client,
    name="AgenteLocal",
    instructions="Eres un profesor universitario. Responde de forma académica y en español.",
)
r1 = await agente_local.run(pregunta)

# Patrón 2: Agente gestionado (instrucciones en Foundry)
r2 = await foundry_agent.run(pregunta)

print("📊 Comparación lado a lado:\n")
print(f"🔧 FoundryChatClient (local):")
txt1 = r1.text[:200] + "..." if len(r1.text) > 200 else r1.text
print(f"   {txt1}\n")
print(f"☁️  FoundryAgent (gestionado):")
txt2 = r2.text[:200] + "..." if len(r2.text) > 200 else r2.text
print(f"   {txt2}")

📊 Comparación lado a lado:

🔧 FoundryChatClient (local):
   Un agente de inteligencia artificial (IA) es un sistema computacional diseñado para percibir su entorno, procesar información y tomar decisiones o realizar acciones de manera autónoma para alcanzar ob...

☁️  FoundryAgent (gestionado):
   Un agente de IA es un programa desarrollado con inteligencia artificial que puede percibir su entorno, procesar información y realizar acciones para cumplir objetivos específicos.


## 🎯 Resumen

| Capacidad | `FoundryChatClient` | `FoundryAgent` |
|-----------|--------------------|-----------------|
| Instrucciones | En tu código Python | En Foundry (portal/API) |
| Tools | Agregados en código (`tools=[...]`) | Configurados en Foundry |
| Sesiones | `AgentSession()` local | Gestionadas por Foundry |
| Control | Total — tú controlas todo | Parcial — Foundry es la fuente de verdad |
| Caso de uso | Prototipado, apps custom | Producción, agentes compartidos |
| Instalación | `pip install agent-framework-foundry` | Mismo paquete |

### Herramientas disponibles en Foundry

| Herramienta | Método | Estado |
|-------------|--------|--------|
| Búsqueda web | `get_web_search_tool()` | GA |
| Code interpreter | `get_code_interpreter_tool()` | GA |
| Búsqueda de archivos | `get_file_search_tool()` | GA |
| Generación de imágenes | `get_image_generation_tool()` | GA |
| Servidores MCP | `get_mcp_tool()` | GA |
| Azure AI Search | `get_azure_ai_search_tool()` | Experimental |
| Bing Grounding | `get_bing_grounding_tool()` | Experimental |

### Regla general

> Si necesitas cambiar instrucciones o tools por ejecución → usa `FoundryChatClient`.
> Si el agente es fijo y solo necesitas invocarlo → usa `FoundryAgent`.

📚 **Más info:** [Doc oficial — Microsoft Foundry Provider](https://learn.microsoft.com/en-us/agent-framework/agents/providers/microsoft-foundry?pivots=programming-language-python)